工具的调用方式
1 直接调用

In [14]:
from langchain_core.tools import tool
@tool
def getweather(city:str)->str:
    '''


    param city: 例如"北京"
    return:天气信息字符串，比如“明天天气阴转小雨，气温12-19摄氏度，出门记得带伞”
    '''
    return "北京2026.9.14 明天天气阴转小雨，气温12-19摄氏度，出门记得带伞~~~"
result=getweather.invoke("北京")
print(result)

北京2026.9.14 明天天气阴转小雨，气温12-19摄氏度，出门记得带伞~~~


方式2 基于模型的调用

In [17]:
from langchain.chat_models import init_chat_model
from rich import print as rprint
##统一使用init_chat_model（）完成模型的初始化
import os
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
#1 读取，env文件中的配置信息，相关的环境变量以.env文件中的优先
load_dotenv(override=True)
my_api_key=os.getenv("API_KEY")
my_base_url=os.getenv("BASE_URL")
#2 初始化大模型
model= init_chat_model(
    model="qwen3.8-max",
    model_provider="openai",
    api_key=my_api_key,
    base_url=my_base_url,
    timeout=30,
    max_retries=3,
)
#3 完成工具和大模型的绑定
model_with_tools=model.bind_tools(
    [getweather]
)
messages=[HumanMessage(content="北京明天的天气如何")]
#4 让大模型生成调用工具的请求
response=model_with_tools.invoke(messages)
#5 添加AImessage
messages.append(response)
tools_calls=response.tool_calls

for tool_call in tools_calls:
    if tool_call["name"]=="getweather":
        tool_response=getweather.invoke(tool_call)
        print(type(tool_response))
        messages.append(tool_response)
print("============================>message<=============================")
for msg in messages:
    msg.pretty_print()
print("============================>message<=============================")
final_response=model_with_tools.invoke(messages)
print(f"final_response:\n{final_response}")

<class 'langchain_core.messages.tool.ToolMessage'>
============================>message<=============================
================================ Human Message =================================

北京明天的天气如何
================================== Ai Message ==================================
Tool Calls:
  getweather (call_07cf77d2d9074b0eb6a1b0fb)
 Call ID: call_07cf77d2d9074b0eb6a1b0fb
  Args:
    city: 北京
================================= Tool Message =================================
Name: getweather

北京2026.9.14 明天天气阴转小雨，气温12-19摄氏度，出门记得带伞~~~
============================>message<=============================
final_response:
content='北京明天（2026年9月14日）的天气情况：\n\n🌥️ **天气**：阴转小雨\n🌡️ **气温**：12～19摄氏度\n\n温馨提示：明天有小雨，出门记得**带伞**哦～另外气温偏低，早晚较凉，建议加件外套，注意保暖！' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 91, 'prompt_tokens': 423, 'total_tokens': 514, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens'